In [ ]:
# ===== 从 X-IIoTID 提取 class1 多分类融合 CSV =====
# 输入: X-IIoTID dataset.csv
# 输出: xiiotid_class1_fusion.csv（供主实验直接读取）
# 说明:
#   - 过滤样本数 < MIN_CLASS 的稀有类，保证 Macro-F1 稳定
#   - 特征按 8 协议组切块 + Protocol/Service 激活标志
#   - 划分: 按类别分层 + 时序小块交错分配 


from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

SRC = Path('X-IIoTID dataset.csv')
OUT = Path('xiiotid_class1_fusion.csv')
PROTOCOL_GROUPS = ['arp', 'icmp', 'http', 'tcp', 'udp', 'dns', 'mqtt', 'mbtcp']
MIN_CLASS = 1000
SPLIT_SEED = 42
SPLIT_RATIOS = (0.70, 0.15, 0.15)
N_BLOCKS = 20

print('loading', SRC)
df = pd.read_csv(SRC, low_memory=False, on_bad_lines='skip')
print('raw', len(df), 'class1', df['class1'].nunique())

vc = df['class1'].value_counts()
keep = vc[vc >= MIN_CLASS].index
df = df[df['class1'].isin(keep)].copy()
print(f'after filter>={MIN_CLASS}:', len(df), 'classes', df['class1'].nunique())
print(df['class1'].value_counts().to_dict())

meta = {'class1', 'class2', 'class3', 'Date', 'Timestamp', 'Scr_IP', 'Des_IP',
        'Protocol', 'Service', 'Conn_state'}
feat_cols = [c for c in df.columns if c not in meta]
for c in feat_cols:
    if df[c].dtype == object:
        s = df[c].astype(str).str.strip().str.upper()
        mapped = s.map({'TRUE': 1, 'FALSE': 0, 'T': 1, 'F': 0, 'YES': 1, 'NO': 0, '1': 1, '0': 0})
        if mapped.notna().mean() > 0.9:
            df[c] = pd.to_numeric(mapped, errors='coerce')
        else:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    elif df[c].dtype == bool:
        df[c] = df[c].astype(np.float32)

for c in ('Scr_port', 'Des_port'):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

num_feats = [c for c in feat_cols if pd.api.types.is_numeric_dtype(df[c])]
print('num_feats', len(num_feats))
X = df[num_feats].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32).values

ts = pd.to_numeric(df['Timestamp'], errors='coerce').fillna(0).values
ips = df['Scr_IP'].astype(str).values
uniq = {ip: i for i, ip in enumerate(sorted(set(ips)))}
sid = np.array([uniq[ip] for ip in ips], dtype=np.int64)

order = np.lexsort((ts, sid))
X, sid, ts = X[order], sid[order], ts[order]
y = df['class1'].values[order]
proto = df['Protocol'].astype(str).str.lower().values[order]
serv = df['Service'].astype(str).str.lower().values[order]

chunks = np.array_split(np.arange(X.shape[1]), len(PROTOCOL_GROUPS))
out = {'stream_id': sid, 'Timestamp': ts, 'final_type': y}
for gi, g in enumerate(PROTOCOL_GROUPS):
    for j, fi in enumerate(chunks[gi]):
        out[f'{g}_f{j}'] = X[:, fi]

proto_map = {'icmp': 'icmp', 'tcp': 'tcp', 'udp': 'udp', 'arp': 'arp'}
serv_map = {'http': 'http', 'https': 'http', 'dns': 'dns', 'mqtt': 'mqtt',
            'modbus': 'mbtcp', 'coap': 'mqtt'}
for g in PROTOCOL_GROUPS:
    out[f'{g}_active'] = np.zeros(len(X), dtype=np.float32)
for i, (p, s) in enumerate(zip(proto, serv)):
    g = proto_map.get(p)
    if g:
        out[f'{g}_active'][i] = 1.0
    g2 = serv_map.get(s)
    if g2:
        out[f'{g2}_active'][i] = 1.0

outdf = pd.DataFrame(out)

# --- 类内时序小块交错划分（缓解整段 70/15/15 的分布偏移）---
split = np.empty(len(outdf), dtype=object)
y_arr = outdf['final_type'].to_numpy()
rng = np.random.RandomState(SPLIT_SEED)
r_tr, r_va, _ = SPLIT_RATIOS

for cls in sorted(outdf['final_type'].unique(), key=str):
    idx = np.where(y_arr == cls)[0]
    n = len(idx)
    if n < 3:
        split[idx] = 'train'
        continue
    n_blocks = min(N_BLOCKS, n)
    edges = np.linspace(0, n, n_blocks + 1, dtype=int)
    blocks = [idx[edges[i]:edges[i + 1]] for i in range(n_blocks) if edges[i + 1] > edges[i]]
    n_blocks = len(blocks)
    n_tr = max(1, int(round(n_blocks * r_tr)))
    n_va = max(1, int(round(n_blocks * r_va)))
    if n_tr + n_va >= n_blocks:
        n_va = max(1, (n_blocks - 1) // 4)
        n_tr = max(1, n_blocks - n_va - 1)
    assign = (['train'] * n_tr + ['val'] * n_va + ['test'] * (n_blocks - n_tr - n_va))
    rng.shuffle(assign)
    for block, name in zip(blocks, assign):
        split[block] = name

outdf['split'] = split
outdf['final_label'] = outdf['final_type']
outdf['seq_id'] = np.arange(len(outdf))

le = LabelEncoder()
le.fit(outdf['final_type'])
print('split', outdf['split'].value_counts().to_dict())
print('split ratios', {k: round(v / len(outdf), 4) for k, v in outdf['split'].value_counts().items()})
print('\n=== final_type x split ===')
ct = pd.crosstab(outdf['final_type'], outdf['split'])
for col in ('train', 'val', 'test'):
    if col not in ct.columns:
        ct[col] = 0
ct = ct[['train', 'val', 'test']]
ct['all'] = ct.sum(axis=1)
print(ct.sort_values('all', ascending=False).to_string())
missing = []
for sp in ('train', 'val', 'test'):
    sub = outdf.loc[outdf['split'] == sp, 'final_type']
    miss = sorted(set(le.classes_) - set(sub.unique()))
    print(f'{sp} classes {sub.nunique()}/{len(le.classes_)}', ('MISSING ' + str(miss)) if miss else 'OK')
    if miss:
        missing.append((sp, miss))
if missing:
    raise RuntimeError(f'split still missing classes: {missing}')

outdf.to_csv(OUT, index=False)
print('saved', OUT.resolve(), 'rows', len(outdf), 'cols', outdf.shape[1])
print('classes', list(le.classes_))
print('提示: 重跑本格后请将主实验 EVAL_ONLY=False 重新训练')


## UltraLite（Mean-Teacher 自蒸馏，可单独运行）




In [ ]:


import os
import math
import shutil
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from torch_geometric.nn import GATv2Conv

_ROOT = os.path.abspath(os.getcwd())
if not os.path.isfile(os.path.join(_ROOT, 'xiiotid_class1_fusion.csv')):
    _ROOT = r'E:\apt\Bo\first'
    os.chdir(_ROOT)
_TMP = os.path.join(_ROOT, '_tmp')
os.makedirs(_TMP, exist_ok=True)
for _k in ('TEMP', 'TMP', 'TMPDIR'):
    os.environ[_k] = _TMP
os.environ['TORCH_HOME'] = os.path.join(_TMP, 'torch')
os.environ['MPLCONFIGDIR'] = os.path.join(_TMP, 'matplotlib')
os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
tempfile.tempdir = _TMP

LITE_CKPT = os.path.join(_ROOT, 'best_gat_mamba_ultralite.pth')
LITE_RESULT = os.path.join(_ROOT, 'best_gat_mamba_ultralite_results.txt')
LITE_CM = os.path.join(_ROOT, 'confusion_matrix_ultralite_test.png')
DATA_CSV = os.path.join(_ROOT, 'xiiotid_class1_fusion.csv')
LAM_SELF_KD = 0.65      # Mean Teacher 软标签权重；硬损失权重 = 1 - LAM_SELF_KD
LAM_F1 = 0.15
EMA_DECAY = 0.999
RAMP_EPOCHS = 10
KD_TEMP = 2.0           

PROTOCOL_GROUPS = ['arp', 'icmp', 'http', 'tcp', 'udp', 'dns', 'mqtt', 'mbtcp']
META_COLS = ('final_label', 'final_type', 'label_encoded', 'seq_id', 'stream_id', 'split', 'Timestamp')

class EdgeIIoTFusionDataset(Dataset):
    """X-IIoTID 协议组滑窗（按 stream 切窗 + 纯度过滤），接口对齐参考实现。"""
    def __init__(self, df, window_size=48, step=6, scaler=None, fit_scaler=False,
                 max_windows=None, min_purity=0.45, normal_idx=None):
        self.window_size = window_size
        self.step = step
        self.min_purity = min_purity
        self.normal_idx = normal_idx
        self.device_dims = []
        self.all_feat_cols = []
        df = df.copy().reset_index(drop=True)
        uniq = {old: i for i, old in enumerate(sorted(df['stream_id'].unique()))}
        df['stream_id'] = df['stream_id'].map(uniq).astype(np.int64)

        for proto in PROTOCOL_GROUPS:
            cols = [c for c in df.columns
                    if (c == proto or c.startswith(proto + '_')) and c not in META_COLS]
            if not cols:
                cols = [f'{proto}_placeholder']
                df[cols[0]] = np.float32(0.0)
            self.device_dims.append(len(cols))
            self.all_feat_cols.extend(cols)

        X_all = df[self.all_feat_cols].values.astype(np.float32)
        y_all = df['label_encoded'].values.astype(np.int64)
        sid_all = df['stream_id'].values.astype(np.int64)

        if scaler is None and fit_scaler:
            self.scaler = StandardScaler()
            X_all = self.scaler.fit_transform(X_all)
        elif scaler is not None:
            self.scaler = scaler
            X_all = self.scaler.transform(X_all)
        else:
            self.scaler = None
        X_all = np.nan_to_num(np.clip(X_all, -8, 8), nan=0.0).astype(np.float32)

        samples, labels = [], []
        for sid in np.unique(sid_all):
            mask = sid_all == sid
            X, y = X_all[mask], y_all[mask]
            if len(X) < window_size:
                continue
            for i in range(0, len(X) - window_size + 1, step):
                yw = y[i:i + window_size]
                vals, cnts = np.unique(yw, return_counts=True)
                frac = {int(v): float(c) / len(yw) for v, c in zip(vals, cnts)}
                atk = [(f, c) for c, f in frac.items()
                       if self.normal_idx is None or c != self.normal_idx]
                # 提高攻击纯度阈值，减少“半 Normal 窗”被标成稀有攻击（如 fuzzing）
                if atk and max(atk)[0] >= 0.55:
                    lab, purity = max(atk)[1], max(atk)[0]
                else:
                    top = int(cnts.max())
                    tied = vals[cnts == top]
                    lab = int(tied[0]) if len(tied) == 1 else int(yw[-1])
                    purity = float(top) / len(yw)
                if purity < min_purity:
                    continue
                samples.append(X[i:i + window_size])
                labels.append(lab)

        if not samples:
            raise ValueError('no windows')

        if max_windows is not None and len(samples) > max_windows:
            # sqrt 比例采样：保留大类多样性，同时给稀有类保底，避免等额过采样导致稀有类假阳
            rng = np.random.RandomState(0)
            labels_arr = np.asarray(labels)
            classes = np.unique(labels_arr)
            raw = np.array([(labels_arr == c).sum() for c in classes], dtype=np.float64)
            alloc = np.sqrt(raw)
            alloc = alloc / alloc.sum() * max_windows
            floor = max(32, max_windows // (len(classes) * 8))
            alloc = np.maximum(np.floor(alloc), floor).astype(np.int64)
            while alloc.sum() > max_windows:
                donors = np.where(alloc > floor)[0]
                if len(donors) == 0:
                    break
                alloc[donors[np.argmax(alloc[donors])]] -= 1
            keep = []
            for c, n_keep in zip(classes, alloc):
                idx = np.where(labels_arr == c)[0]
                if len(idx) > n_keep:
                    idx = rng.choice(idx, int(n_keep), replace=False)
                keep.extend(idx.tolist())
            if len(keep) > max_windows:
                keep = rng.choice(keep, max_windows, replace=False).tolist()
            samples = [samples[i] for i in keep]
            labels = [labels[i] for i in keep]

        self.samples = np.stack(samples)
        self.labels = np.asarray(labels, dtype=np.int64)
        print(f'windows={len(self)} dims={self.device_dims} step={step} '
              f'classes={len(np.unique(self.labels))}')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.samples[idx]), torch.tensor(self.labels[idx], dtype=torch.long)


class SimpleMamba(nn.Module):
    def __init__(self, d_model, d_hidden, kernel=4):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.gate = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.out_proj = nn.Linear(d_hidden, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x_t, L = x.transpose(1, 2), x.size(1)
        h = (self.conv(x_t)[:, :, :L] * torch.sigmoid(self.gate(x_t)[:, :, :L])).transpose(1, 2)
        return self.norm(self.out_proj(h) + x)



class EMA:
    def __init__(self, model, decay=0.995):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def focal_loss(logits, labels, alpha=0.25, gamma=2.0, weight=None):
    ce = F.cross_entropy(logits.float(), labels, reduction='none',
                         weight=weight, label_smoothing=0.015)
    pt = torch.exp(-ce.detach())
    return (alpha * (1 - pt).clamp(0, 1) ** gamma * ce).mean()


def soft_f1_loss(logits, y, n_cls, eps=1e-6):
    p = F.softmax(logits.float(), 1)
    t = F.one_hot(y, n_cls).float()
    tp = (p * t).sum(0)
    fp = (p * (1 - t)).sum(0)
    fn = ((1 - p) * t).sum(0)
    f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return 1 - f1.mean()


@torch.no_grad()
def collect_logits(model, loader, device):
    model.eval()
    logits_all, trues = [], []
    for x, y in loader:
        x = x.to(device)
        logits_all.append(model(x).float().cpu())
        trues.append(y)
    return torch.cat(logits_all, 0), torch.cat(trues, 0)


def macro_fpr_score(y_true, y_pred, n_cls):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fprs = []
    for c in range(n_cls):
        fp = np.sum((y_pred == c) & (y_true != c))
        neg = np.sum(y_true != c)
        fprs.append(float(fp) / max(int(neg), 1))
    return float(np.mean(fprs))


def apply_fpr_decision(logits, adj_tau=0.0, log_prior=None, logit_bias=None, pair_rules=None):

    out = logits
    if log_prior is not None and adj_tau != 0:
        out = out - float(adj_tau) * log_prior
    if logit_bias is not None:
        out = out + logit_bias
    pred = out.argmax(1)
    if not pair_rules:
        return pred
    top2 = out.topk(2, dim=1).indices
    pred = pred.clone()
    for rule in pair_rules:
        prefer = int(rule['prefer'])
        suppress = int(rule['suppress'])
        delta = float(rule['delta'])
        in_top2 = (top2 == prefer).any(1) & (top2 == suppress).any(1)
        margin = out[:, suppress] - out[:, prefer]
        flip = (pred == suppress) & in_top2 & (margin <= delta)
        pred[flip] = prefer
    return pred


def tune_logit_adjustment(logits, y_true, class_prior, labels, class_names=None):
   
    n_cls = len(labels)
    log_prior = torch.log(torch.clamp(class_prior.cpu().float(), min=1e-6)).view(1, -1)
    y_np = y_true.numpy() if torch.is_tensor(y_true) else np.asarray(y_true)
    logits = logits.cpu().float()
    names = list(class_names) if class_names is not None else [str(i) for i in range(n_cls)]
    name_to_i = {n: i for i, n in enumerate(names)}

    def pack(tau, rules):
        pred = apply_fpr_decision(logits, adj_tau=tau, log_prior=log_prior, pair_rules=rules).numpy()
        f1 = f1_score(y_np, pred, average='macro', labels=labels, zero_division=0)
        fpr = macro_fpr_score(y_np, pred, n_cls)
        return float(f1), float(fpr), pred

    base_f1, base_fpr, base_pred = pack(0.0, None)
    f1_floor = base_f1 * 0.99
    cm = confusion_matrix(y_np, base_pred, labels=list(range(n_cls)))
    pair_cands = []
    for j in range(n_cls):
        for i in range(n_cls):
            if i == j:
                continue
            fp_ij = int(cm[i, j])
            if fp_ij >= 3:
                pair_cands.append((fp_ij, i, j))
   
    if 'Generic_scanning' in name_to_i and 'Scanning_vulnerability' in name_to_i:
        gs, sv = name_to_i['Generic_scanning'], name_to_i['Scanning_vulnerability']
        if not any((i == gs and j == sv) for _, i, j in pair_cands):
            pair_cands.append((int(cm[gs, sv]), gs, sv))
    pair_cands.sort(reverse=True)

    deltas = np.concatenate([np.linspace(0.0, 6.0, 31), np.array([1e6])])
    rules = []
    best_f1, best_fpr = base_f1, base_fpr
    for _, prefer, suppress in pair_cands:
        curve = []
        for d in deltas:
            trial = rules + [{'prefer': prefer, 'suppress': suppress, 'delta': float(d)}]
            f1, fpr, _ = pack(0.0, trial)
            if f1 >= f1_floor:
                curve.append((float(d), fpr, f1))
        if not curve:
            continue
        min_fpr = min(x[1] for x in curve)
        if min_fpr >= best_fpr - 1e-12:
            continue
        target = min_fpr + 0.10 * (best_fpr - min_fpr)  
        chosen = None
        for d, fpr, f1 in curve:
            if fpr <= target + 1e-15:
                chosen = (d, fpr, f1)
                break
        if chosen is None:
            continue
        d, fpr, f1 = chosen
        rules.append({'prefer': int(prefer), 'suppress': int(suppress), 'delta': float(d),
                      'prefer_name': names[prefer], 'suppress_name': names[suppress]})
        best_f1, best_fpr = f1, fpr

    print(f'  calib: val FPR {base_fpr*1e3:.3f}e-3 -> {best_fpr*1e3:.3f}e-3  '
          f'F1 {base_f1:.4f} -> {best_f1:.4f}  rules={len(rules)}')
    for r in rules:
        print(f"    {r['suppress_name']} -> {r['prefer_name']}  delta={r['delta']:.3f}")
    return 0.0, None, rules, best_f1


@torch.no_grad()
def evaluate(model, loader, device, label_encoder, desc='Eval',
             class_prior=None, adj_tau=0.0, logit_bias=None, pair_rules=None):
    model.eval()
    preds, trues = [], []
    log_prior = None
    if class_prior is not None and adj_tau != 0:
        log_prior = torch.log(torch.clamp(class_prior, min=1e-6)).to(device).view(1, -1)
    bias = None
    if logit_bias is not None:
        bias = torch.as_tensor(logit_bias, dtype=torch.float32, device=device).view(1, -1)
    for x, y in tqdm(loader, desc=desc, leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x).float()
        pred = apply_fpr_decision(
            logits, adj_tau=adj_tau, log_prior=log_prior,
            logit_bias=bias, pair_rules=pair_rules)
        preds.extend(pred.cpu().numpy())
        trues.extend(y.cpu().numpy())
    all_labels = list(range(len(label_encoder.classes_)))
    names = list(label_encoder.classes_)
    report = classification_report(
        trues, preds, labels=all_labels, target_names=names, digits=4, zero_division=0)
    f1 = f1_score(trues, preds, average='macro', labels=all_labels, zero_division=0)
    acc = float(accuracy_score(trues, preds))
    return {
        'report': report,
        'macro_f1': float(f1),
        'accuracy': acc,
        'y_true': np.asarray(trues),
        'y_pred': np.asarray(preds),
        'macro_fpr': macro_fpr_score(trues, preds, len(all_labels)),
    }


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix (Count)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='RdYlGn',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], vmin=0, vmax=100)
    axes[1].set_title('Confusion Matrix (Normalized %)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'confusion matrix saved: {save_path}')


class InnovativeGATWithMambaUltraLite(nn.Module):
    def __init__(self, device_dims, d_model=16, d_hidden=32, num_classes=16,
                 window_size=48, gat_heads=1, dropout=0.1):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.device_projectors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, d_model), nn.LayerNorm(d_model), nn.GELU())
            for dim in device_dims
        ])

        self.share_logit = nn.Parameter(torch.tensor(-2.0))
        self.act_temp = nn.Parameter(torch.tensor(1.0))
        prior_weight = torch.full((self.num_devices, self.num_devices), 0.1)
        for i, j in [(2, 3), (3, 2), (3, 4), (4, 3), (4, 5), (5, 4), (6, 3), (3, 6), (7, 3), (3, 7)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 1.0
        for i, j in [(0, 1), (1, 0), (0, 3), (3, 0), (1, 3), (3, 1)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 0.2
        self.register_buffer('prior_weight', prior_weight)
        self.dynamic_prior_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, 1), nn.Sigmoid())
        self.gat1 = GATv2Conv(d_model, d_model, heads=gat_heads, concat=False,
                              dropout=dropout, add_self_loops=False, edge_dim=1)
        self.n1 = nn.LayerNorm(d_model)
        mamba_dim = d_model * self.num_devices
        self.mamba = SimpleMamba(mamba_dim, d_hidden, kernel=3)
        self.attn = nn.Linear(mamba_dim, 1)
        # Head A (A / 1e-3 / bs64 / plateau)
        self.classifier = nn.Sequential(
            nn.LayerNorm(mamba_dim * 2),
            nn.Linear(mamba_dim * 2, num_classes))

    def calibrate_cross_protocol(self, device_feats):
        """活性加权共识 + 软融合。device_feats: (B,T,N,D)"""
        act = device_feats.norm(dim=-1, keepdim=True)  # (B,T,N,1)
        temp = self.act_temp.clamp(0.2, 5.0)
        w = torch.softmax(act / temp, dim=2)
        shared = (device_feats * w).sum(dim=2, keepdim=True)
        gamma = torch.sigmoid(self.share_logit)
        return (1.0 - gamma) * device_feats + gamma * shared

    def compute_dynamic_edges(self, pooled):
        B, N, D = pooled.shape
        device = pooled.device
        ei = torch.tensor(
            [[i, j] for i in range(N) for j in range(N) if i != j],
            dtype=torch.long, device=device).t().contiguous()
        normed = F.normalize(pooled, p=2, dim=-1)
        sim = torch.matmul(normed, normed.transpose(1, 2))
        dyn = torch.stack([sim[:, i, j] for i, j in zip(ei[0], ei[1])], dim=-1)
        fi, fj = pooled[:, ei[0]], pooled[:, ei[1]]
        alpha = self.dynamic_prior_gate(torch.cat([fi, fj], -1)).squeeze(-1)
        prior = self.prior_weight[ei[0], ei[1]].view(1, -1)
      
        agree = ((fi * fj).sum(-1) / (fi.norm(dim=-1) * fj.norm(dim=-1) + 1e-6)).clamp(0, 1)
        ew = ((alpha * prior + (1 - alpha) * dyn) * (0.5 + 0.5 * agree)).reshape(-1, 1)
        off = torch.arange(B, device=device) * N
        eib = (ei.unsqueeze(1) + off.view(1, -1, 1)).reshape(2, -1)
        return eib, ew

    def forward(self, x):
        B, T, _ = x.shape
        device_feats, split_idx = [], 0
        for proj, dim in zip(self.device_projectors, self.device_dims):
            device_feats.append(proj(x[:, :, split_idx:split_idx + dim]))
            split_idx += dim
        device_feats = torch.stack(device_feats, dim=2)
        device_feats = self.calibrate_cross_protocol(device_feats)
        pooled = device_feats.mean(1)
        edge_index, edge_weights = self.compute_dynamic_edges(pooled)
        h = pooled.reshape(-1, self.d_model)
        h = self.n1(h + self.gat1(h, edge_index, edge_attr=edge_weights))
        out_seq = device_feats + h.reshape(B, 1, self.num_devices, self.d_model)
        fused = self.mamba(out_seq.reshape(B, T, -1))
        w = torch.softmax(self.attn(fused).squeeze(-1), dim=1)
        feat = torch.cat([
            (fused * w.unsqueeze(-1)).sum(1), fused.mean(1),
        ], dim=-1)
        return self.classifier(feat)


def kd_loss(student_logits, teacher_logits, T=2.0):

    s = F.log_softmax(student_logits.float() / T, dim=1)
    t = F.softmax(teacher_logits.float() / T, dim=1)
    return F.kl_div(s, t, reduction='batchmean') * (T * T)


def train_lite_epoch(student, loader, optimizer, device, n_cls,
                     class_weight, ema, lam_kd=0.65, lam_f1=0.15, kd_temp=1.0):
   
    student.train()
    total, all_preds, all_labels = 0.0, [], []
    pbar = tqdm(loader, desc='UltraTrain', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
    
        with torch.no_grad():
            bak = {k: v.detach().clone() for k, v in student.state_dict().items()}
            ema.copy_to(student)
            student.eval()
            t_logits = student(x)
            student.train()
            student.load_state_dict(bak, strict=True)
        s_logits = student(x)
        hard = focal_loss(s_logits, y, weight=class_weight) + lam_f1 * soft_f1_loss(s_logits, y, n_cls)
        loss = (1 - lam_kd) * hard + lam_kd * kd_loss(s_logits, t_logits, T=kd_temp) if lam_kd > 0 else hard
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optimizer.step()
        ema.update(student)
        bs = x.size(0)
        total += loss.item() * bs
        all_preds.extend(s_logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        pbar.set_postfix(loss=float(loss.item()))
    n = max(len(loader.dataset), 1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total / n, f1



WINDOW_SIZE, TRAIN_STEP, EVAL_STEP = 48, 6, 48
MAX_TRAIN, MAX_EVAL = 40000, 12000

CLF_TAG, DROPOUT, SCHEDULER_NAME = 'A', 0.1, 'plateau'
BATCH_SIZE, EPOCHS, LR, PATIENCE = 64, 40, 1e-3, 8
FORCE_FRESH = False  
EVAL_ONLY = False    
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'UltraLite  device={DEVICE}  MeanTeacher lam_kd={LAM_SELF_KD} ema={EMA_DECAY} T={KD_TEMP}')
print(f'hp: clf={CLF_TAG} lr={LR} dropout={DROPOUT} batch={BATCH_SIZE} sch={SCHEDULER_NAME} patience={PATIENCE}')
print(f'save={LITE_CKPT}')
if FORCE_FRESH and os.path.isfile(LITE_CKPT):
    bak = LITE_CKPT.replace('.pth', '_bak.pth')
    shutil.copy2(LITE_CKPT, bak)
    os.remove(LITE_CKPT)
    print(f'FORCE_FRESH: backed up old ckpt -> {bak}')
if FORCE_FRESH and os.path.isfile(LITE_RESULT):
    bak_r = LITE_RESULT.replace('.txt', '_bak.txt')
    shutil.copy2(LITE_RESULT, bak_r)
    print(f'FORCE_FRESH: backed up old results -> {bak_r}')
assert os.path.isfile(DATA_CSV), '找不到 xiiotid_class1_fusion.csv'

df = pd.read_csv(DATA_CSV, low_memory=False)
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df['final_type'])
n_cls = len(label_encoder.classes_)
normal_idx = int(np.where(label_encoder.classes_ == 'Normal')[0][0]) if 'Normal' in set(label_encoder.classes_) else None
df_train = df[df.split == 'train'].reset_index(drop=True)
df_val = df[df.split == 'val'].reset_index(drop=True)
df_test = df[df.split == 'test'].reset_index(drop=True)

train_ds = EdgeIIoTFusionDataset(
    df_train, window_size=WINDOW_SIZE, step=TRAIN_STEP, fit_scaler=True,
    max_windows=MAX_TRAIN, min_purity=0.50, normal_idx=normal_idx)
val_ds = EdgeIIoTFusionDataset(
    df_val, window_size=WINDOW_SIZE, step=EVAL_STEP, scaler=train_ds.scaler,
    max_windows=MAX_EVAL, min_purity=0.50, normal_idx=normal_idx)
test_ds = EdgeIIoTFusionDataset(
    df_test, window_size=WINDOW_SIZE, step=EVAL_STEP, scaler=train_ds.scaler,
    max_windows=MAX_EVAL, min_purity=0.50, normal_idx=normal_idx)

counts = np.maximum(np.bincount(train_ds.labels, minlength=n_cls).astype(np.float64), 1.0)
sw = 1.0 / np.power(counts[train_ds.labels], 0.35)
sw = sw / sw.mean()
sampler = torch.utils.data.WeightedRandomSampler(torch.DoubleTensor(sw), len(train_ds), True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
cw = np.clip(np.sqrt(counts.sum() / (n_cls * counts)), 0.5, 2.0)
class_weight = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
class_prior = torch.tensor(counts / counts.sum(), dtype=torch.float32)

student = InnovativeGATWithMambaUltraLite(
    device_dims=train_ds.device_dims, d_model=16, d_hidden=32,
    num_classes=n_cls, window_size=WINDOW_SIZE, gat_heads=1, dropout=DROPOUT,
).to(DEVICE)
n_params = sum(p.numel() for p in student.parameters())
print(f'ultralite params={n_params/1e6:.3f}M  | MeanTeacher ON | activity-weighted soft share')

ema = EMA(student, decay=EMA_DECAY)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=2e-4)
if SCHEDULER_NAME == 'plateau':
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2)
elif SCHEDULER_NAME == 'step':
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.5)
else:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=8, T_mult=2, eta_min=1e-6)

best_val_f1, best_epoch, bad = -1.0, 0, 0
start_epoch = 1
if (not FORCE_FRESH) and os.path.isfile(LITE_CKPT):
    _resume = torch.load(LITE_CKPT, map_location=DEVICE, weights_only=False)
    student.load_state_dict(_resume['model_state'], strict=True)
    ema.shadow = {k: v.detach().clone().to(DEVICE) for k, v in _resume['model_state'].items()}
    best_val_f1 = float(_resume.get('val_macro_f1', -1.0))
    best_epoch = int(_resume.get('epoch', 0))
    start_epoch = best_epoch + 1
    print(f'RESUME from {LITE_CKPT} epoch={best_epoch} val_f1={best_val_f1:.4f} -> continue {start_epoch}..{EPOCHS}')
    del _resume
if EVAL_ONLY:
    start_epoch = EPOCHS + 1
    print('EVAL_ONLY: skip training; FPR-calibrate existing weights (no extra params)')
if start_epoch > EPOCHS:
    print(f'already finished epochs (start={start_epoch}>={EPOCHS}); jump to final eval')
def consistency_rampup(epoch, ramp_epochs=RAMP_EPOCHS, max_w=LAM_SELF_KD):
    """Mean Teacher (Tarvainen & Valpola): soft-label weight ramp-up."""
    t = 1.0 if ramp_epochs <= 0 else min(1.0, float(epoch) / float(ramp_epochs))
    return float(max_w * math.exp(-5.0 * (1.0 - t) ** 2))

for epoch in range(start_epoch, EPOCHS + 1):
    lam_now = consistency_rampup(epoch)
    print(f'\n--- UltraLite Epoch {epoch}/{EPOCHS} --- lam_kd={lam_now:.3f}')
    tr_loss, tr_f1 = train_lite_epoch(
        student, train_loader, optimizer, DEVICE, n_cls, class_weight, ema,
        lam_kd=lam_now, lam_f1=LAM_F1, kd_temp=KD_TEMP)
    bak = {k: v.detach().clone() for k, v in student.state_dict().items()}
    ema.copy_to(student)
    val = evaluate(student, val_loader, DEVICE, label_encoder, desc='UltraValid')
    student.load_state_dict(bak)
    print(f'Train Loss {tr_loss:.4f} | Train F1 {tr_f1:.4f} | Val Macro-F1 {val["macro_f1"]:.4f}')
    if val['macro_f1'] > best_val_f1 + 1e-4:
        best_val_f1, best_epoch, bad = val['macro_f1'], epoch, 0
        torch.save({
            'model_state': {k: v.cpu().clone() for k, v in ema.shadow.items()},
            'epoch': best_epoch,
            'val_macro_f1': best_val_f1,
            'classes': list(label_encoder.classes_),
            'device_dims': list(train_ds.device_dims),
            'd_model': 16, 'd_hidden': 32, 'gat_heads': 1,
            'dropout': DROPOUT, 'clf': CLF_TAG, 'lr': LR, 'batch_size': BATCH_SIZE,
            'scheduler': SCHEDULER_NAME,
            'model': 'InnovativeGATWithMambaUltraLite',
            'use_self_kd': True,
            'lam_kd': LAM_SELF_KD,
            'ema_decay': EMA_DECAY,
            'ramp_epochs': RAMP_EPOCHS,
            'kd_temp': KD_TEMP,
            'soft_teacher': 'EMA-MeanTeacher',
            'class_prior': class_prior.cpu(),
        }, LITE_CKPT)
        print(f'  >> saved {LITE_CKPT}  Val Macro-F1={best_val_f1:.4f}')
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f'early stop (patience={PATIENCE})')
            break
    if SCHEDULER_NAME == 'plateau':
        scheduler.step(val['macro_f1'])
    else:
        scheduler.step()
    print(f'  lr={optimizer.param_groups[0]["lr"]:.2e}')

ckpt = torch.load(LITE_CKPT, map_location=DEVICE, weights_only=False)
student.load_state_dict(ckpt['model_state'])
val_logits, val_y = collect_logits(student, val_loader, DEVICE)
adj_tau, logit_bias, pair_rules, cal_f1 = tune_logit_adjustment(
    val_logits, val_y, class_prior, list(range(n_cls)),
    class_names=list(label_encoder.classes_))
ckpt['adj_tau'] = float(adj_tau)
ckpt['logit_bias'] = None if logit_bias is None else logit_bias.detach().cpu().float()
ckpt['pair_rules'] = pair_rules
torch.save(ckpt, LITE_CKPT)

val = evaluate(student, val_loader, DEVICE, label_encoder, desc='UltraValid',
               class_prior=class_prior, adj_tau=adj_tau, logit_bias=logit_bias,
               pair_rules=pair_rules)
test = evaluate(student, test_loader, DEVICE, label_encoder, desc='UltraTest',
                class_prior=class_prior, adj_tau=adj_tau, logit_bias=logit_bias,
                pair_rules=pair_rules)
print('\n' + '=' * 60)
print('GAT-Mamba UltraLite  (Mean-Teacher self-distillation)')
print(f'clf={CLF_TAG} lr={LR} drop={DROPOUT} batch={BATCH_SIZE} sch={SCHEDULER_NAME}')
print(f'Best Epoch {best_epoch} | adj_tau={adj_tau:.3f} | params={n_params/1e6:.3f}M (unchanged)')
print(f'Val Macro-F1 {val["macro_f1"]:.4f} | Test Macro-F1 {test["macro_f1"]:.4f} | Acc {test["accuracy"]:.4f}')
print(f'Val macro-FPR {val["macro_fpr"]*1e3:.3f}e-3 | Test macro-FPR {test["macro_fpr"]*1e3:.3f}e-3')
print('=' * 60)
print(test['report'])
plot_confusion_matrix(test['y_true'], test['y_pred'], list(label_encoder.classes_), save_path=LITE_CM)
rules_str = '; '.join(
    f"{r['suppress_name']}->{r['prefer_name']}:d={r['delta']:.3f}" for r in (pair_rules or []))
with open(LITE_RESULT, 'w', encoding='utf-8') as f:
    f.write(f'model=InnovativeGATWithMambaUltraLite\n')
    f.write(f'params_M={n_params/1e6:.6f}\n')
    f.write(f'use_self_kd=True\nlam_kd={LAM_SELF_KD}\nema_decay={EMA_DECAY}\n'
            f'ramp_epochs={RAMP_EPOCHS}\nkd_temp={KD_TEMP}\n')
    f.write(f'clf={CLF_TAG}\nlr={LR}\ndropout={DROPOUT}\nbatch_size={BATCH_SIZE}\nscheduler={SCHEDULER_NAME}\npatience={PATIENCE}\n')
    f.write(f'best_epoch={best_epoch}\nadj_tau={adj_tau:.6f}\n')
    f.write(f'pair_rules={rules_str}\n')
    f.write(f'val_macro_f1={val["macro_f1"]:.6f}\n')
    f.write(f'test_macro_f1={test["macro_f1"]:.6f}\n')
    f.write(f'val_macro_fpr={val["macro_fpr"]:.8f}\n')
    f.write(f'test_macro_fpr={test["macro_fpr"]:.8f}\n')
    f.write('note=Mean-Teacher self-KD + val pair-margin FPR calib (inference-only, 0 extra params)\n\n')
    f.write('===== VAL =====\n' + val['report'] + '\n')
    f.write('===== TEST =====\n' + test['report'] + '\n')
print(f'saved: {LITE_CKPT} | {LITE_RESULT}')


## 三组消融实验（对齐 UltraLite 主实验；流程对齐目录2）



In [ ]:
import matplotlib
matplotlib.use('Agg')
# ===== 三组消融实验（与 UltraLite 主实验严格对齐；流程对齐目录2）=====
# A0: 纯基线（无GAT/无Mamba/无自蒸馏；原始特征mean+线性分类）
# A1: 仅先验动态门控 GAT（Dynamic-Prior GAT + Atten||Mean，硬标签，无 Mamba）
# A2: SimpleMamba + Mean-Teacher 自蒸馏（无 GAT；Atten||Mean）
# 超参对齐主实验: window=48 / step=6/48 / batch=64 / lr=1e-3 / patience=8 / ReduceLROnPlateau

import os
import math
import tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from torch_geometric.nn import GATv2Conv

# ===================== 路径与全局配置 =====================
_CANDS = [
    r'E:\apt\YES\1',
    os.path.abspath(os.getcwd()),
    os.path.dirname(os.path.abspath(os.getcwd())),
    '/root/lllll/1',
    r'E:\apt\Bo\first',
]
_ROOT = None
for _cand in _CANDS:
    if os.path.isfile(os.path.join(_cand, 'xiiotid_class1_fusion.csv')):
        _ROOT = _cand
        os.chdir(_ROOT)
        break
if _ROOT is None:
    raise FileNotFoundError('数据集不存在: ' + ' | '.join(_CANDS))
DATA_CSV = os.path.join(_ROOT, 'xiiotid_class1_fusion.csv')
_TMP = os.path.join(_ROOT, '_tmp')
os.makedirs(_TMP, exist_ok=True)
for _k in ('TEMP', 'TMP', 'TMPDIR'):
    os.environ[_k] = _TMP
os.environ['TORCH_HOME'] = os.path.join(_TMP, 'torch')
os.environ['MPLCONFIGDIR'] = os.path.join(_TMP, 'matplotlib')
os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)
os.makedirs(os.environ['MPLCONFIGDIR'], exist_ok=True)
tempfile.tempdir = _TMP

SUMMARY_TXT = os.path.join(_ROOT, 'ablation_summary.txt')
print(f'输出目录(当前路径)={_ROOT}')
print(f'数据文件={DATA_CSV}')

PROTOCOL_GROUPS = ['arp', 'icmp', 'http', 'tcp', 'udp', 'dns', 'mqtt', 'mbtcp']
META_COLS = ('final_label', 'final_type', 'label_encoded', 'seq_id', 'stream_id', 'split', 'Timestamp')


WINDOW_SIZE, TRAIN_STEP, EVAL_STEP = 48, 6, 48
MAX_TRAIN, MAX_EVAL = 40000, 12000
MIN_PURITY = 0.50
CLF_TAG, DROPOUT, SCHEDULER_NAME = 'A', 0.1, 'plateau'
BATCH_SIZE, EPOCHS, LR, PATIENCE = 64, 40, 1e-3, 8
D_MODEL, D_HIDDEN, GAT_HEADS = 16, 32, 1
LAM_F1 = 0.15
LAM_SELF_KD = 0.65
EMA_DECAY = 0.999
RAMP_EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ABLATION_CONFIGS = [
    dict(
        tag='A0_none', name='A0_纯基线',
        use_gat=False, use_mamba=False, use_mean_teacher=False, pool='mean',
        ckpt='best_ablation_A0_none.pth',
        result='best_ablation_A0_none_results.txt',
        cm='cm_A0_none.png'
    ),
    dict(
        tag='A1_dyn_prior_gat', name='A1_先验动态门控GAT',
        use_gat=True, use_mamba=False, use_mean_teacher=False, pool='attn_mean',
        ckpt='best_ablation_A1_dyn_prior_gat.pth',
        result='best_ablation_A1_dyn_prior_gat_results.txt',
        cm='cm_A1_dyn_prior_gat.png'
    ),
    dict(
        tag='A2_mamba_kd', name='A2_SimpleMamba+自蒸馏',
        use_gat=False, use_mamba=True, use_mean_teacher=True, pool='attn_mean',
        ckpt='best_ablation_A2_mamba_kd.pth',
        result='best_ablation_A2_mamba_kd_results.txt',
        cm='cm_A2_mamba_kd.png'
    ),
]

RETRAIN_TAGS = {'A0_none'}


class EdgeIIoTFusionDataset(Dataset):
    """X-IIoTID 协议组滑窗（按 stream 切窗 + 纯度过滤），与主实验一致。"""
    def __init__(self, df, window_size=48, step=6, scaler=None, fit_scaler=False,
                 max_windows=None, min_purity=0.45, normal_idx=None):
        self.window_size = window_size
        self.step = step
        self.min_purity = min_purity
        self.normal_idx = normal_idx
        self.device_dims = []
        self.all_feat_cols = []
        df = df.copy().reset_index(drop=True)
        uniq = {old: i for i, old in enumerate(sorted(df['stream_id'].unique()))}
        df['stream_id'] = df['stream_id'].map(uniq).astype(np.int64)

        for proto in PROTOCOL_GROUPS:
            cols = [c for c in df.columns
                    if (c == proto or c.startswith(proto + '_')) and c not in META_COLS]
            if not cols:
                cols = [f'{proto}_placeholder']
                df[cols[0]] = np.float32(0.0)
            self.device_dims.append(len(cols))
            self.all_feat_cols.extend(cols)

        X_all = df[self.all_feat_cols].values.astype(np.float32)
        y_all = df['label_encoded'].values.astype(np.int64)
        sid_all = df['stream_id'].values.astype(np.int64)

        if scaler is None and fit_scaler:
            self.scaler = StandardScaler()
            X_all = self.scaler.fit_transform(X_all)
        elif scaler is not None:
            self.scaler = scaler
            X_all = self.scaler.transform(X_all)
        else:
            self.scaler = None
        X_all = np.nan_to_num(np.clip(X_all, -8, 8), nan=0.0).astype(np.float32)

        samples, labels = [], []
        for sid in np.unique(sid_all):
            mask = sid_all == sid
            X, y = X_all[mask], y_all[mask]
            if len(X) < window_size:
                continue
            for i in range(0, len(X) - window_size + 1, step):
                yw = y[i:i + window_size]
                vals, cnts = np.unique(yw, return_counts=True)
                frac = {int(v): float(c) / len(yw) for v, c in zip(vals, cnts)}
                atk = [(f, c) for c, f in frac.items()
                       if self.normal_idx is None or c != self.normal_idx]
                if atk and max(atk)[0] >= 0.55:
                    lab, purity = max(atk)[1], max(atk)[0]
                else:
                    top = int(cnts.max())
                    tied = vals[cnts == top]
                    lab = int(tied[0]) if len(tied) == 1 else int(yw[-1])
                    purity = float(top) / len(yw)
                if purity < min_purity:
                    continue
                samples.append(X[i:i + window_size])
                labels.append(lab)

        if not samples:
            raise ValueError('no windows')

        if max_windows is not None and len(samples) > max_windows:
            rng = np.random.RandomState(0)
            labels_arr = np.asarray(labels)
            classes = np.unique(labels_arr)
            raw = np.array([(labels_arr == c).sum() for c in classes], dtype=np.float64)
            alloc = np.sqrt(raw)
            alloc = alloc / alloc.sum() * max_windows
            floor = max(32, max_windows // (len(classes) * 8))
            alloc = np.maximum(np.floor(alloc), floor).astype(np.int64)
            while alloc.sum() > max_windows:
                donors = np.where(alloc > floor)[0]
                if len(donors) == 0:
                    break
                alloc[donors[np.argmax(alloc[donors])]] -= 1
            keep = []
            for c, n_keep in zip(classes, alloc):
                idx = np.where(labels_arr == c)[0]
                if len(idx) > n_keep:
                    idx = rng.choice(idx, int(n_keep), replace=False)
                keep.extend(idx.tolist())
            if len(keep) > max_windows:
                keep = rng.choice(keep, max_windows, replace=False).tolist()
            samples = [samples[i] for i in keep]
            labels = [labels[i] for i in keep]

        self.samples = np.stack(samples)
        self.labels = np.asarray(labels, dtype=np.int64)
        print(f'windows={len(self)} dims={self.device_dims} step={step} '
              f'classes={len(np.unique(self.labels))}')

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.FloatTensor(self.samples[idx]), torch.tensor(self.labels[idx], dtype=torch.long)


class SimpleMamba(nn.Module):
    def __init__(self, d_model, d_hidden, kernel=3):
        super().__init__()
        self.conv = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.gate = nn.Conv1d(d_model, d_hidden, kernel_size=kernel, padding=kernel - 1)
        self.out_proj = nn.Linear(d_hidden, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x_t, L = x.transpose(1, 2), x.size(1)
        h = (self.conv(x_t)[:, :, :L] * torch.sigmoid(self.gate(x_t)[:, :, :L])).transpose(1, 2)
        return self.norm(self.out_proj(h) + x)


class EMA:
    def __init__(self, model, decay=EMA_DECAY):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


def focal_loss(logits, labels, alpha=0.25, gamma=2.0, weight=None):
    ce = F.cross_entropy(logits.float(), labels, reduction='none',
                         weight=weight, label_smoothing=0.015)
    pt = torch.exp(-ce.detach())
    return (alpha * (1 - pt).clamp(0, 1) ** gamma * ce).mean()


def soft_f1_loss(logits, y, n_cls, eps=1e-6):
    p = F.softmax(logits.float(), 1)
    t = F.one_hot(y, n_cls).float()
    tp = (p * t).sum(0)
    fp = (p * (1 - t)).sum(0)
    fn = ((1 - p) * t).sum(0)
    f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return 1 - f1.mean()


def kd_loss(student_logits, teacher_logits, T=2.0):
    s = F.log_softmax(student_logits.float() / T, dim=1)
    t = F.softmax(teacher_logits.float() / T, dim=1)
    return F.kl_div(s, t, reduction='batchmean') * (T * T)


def consistency_rampup(epoch, ramp_epochs=RAMP_EPOCHS, max_w=LAM_SELF_KD):
    t = 1.0 if ramp_epochs <= 0 else min(1.0, float(epoch) / float(ramp_epochs))
    return float(max_w * math.exp(-5.0 * (1.0 - t) ** 2))


@torch.no_grad()
def collect_logits(model, loader, device):
    model.eval()
    logits_all, trues = [], []
    for x, y in loader:
        x = x.to(device)
        logits_all.append(model(x).float().cpu())
        trues.append(y)
    return torch.cat(logits_all, 0), torch.cat(trues, 0)


def tune_logit_adjustment(logits, y_true, class_prior, labels):
    log_prior = torch.log(torch.clamp(class_prior, min=1e-6))
    best_tau, best_f1 = 0.0, -1.0
    y_np = y_true.numpy()
    for tau in np.linspace(0.0, 1.5, 16):
        pred = (logits - tau * log_prior).argmax(1).numpy()
        f1 = f1_score(y_np, pred, average='macro', labels=labels, zero_division=0)
        if f1 > best_f1:
            best_f1, best_tau = float(f1), float(tau)
    return best_tau, best_f1


@torch.no_grad()
def evaluate(model, loader, device, label_encoder, desc='Eval',
             class_prior=None, adj_tau=0.0):
    model.eval()
    preds, trues = [], []
    log_prior = None
    if class_prior is not None and adj_tau != 0:
        log_prior = torch.log(torch.clamp(class_prior, min=1e-6)).to(device)
    for x, y in tqdm(loader, desc=desc, leave=False):
        x, y = x.to(device), y.to(device)
        logits = model(x).float()
        if log_prior is not None:
            logits = logits - adj_tau * log_prior
        pred = logits.argmax(1)
        preds.extend(pred.cpu().numpy())
        trues.extend(y.cpu().numpy())
    all_labels = list(range(len(label_encoder.classes_)))
    names = list(label_encoder.classes_)
    report = classification_report(
        trues, preds, labels=all_labels, target_names=names, digits=4, zero_division=0)
    f1 = f1_score(trues, preds, average='macro', labels=all_labels, zero_division=0)
    acc = float(accuracy_score(trues, preds))
    return {
        'report': report, 'macro_f1': float(f1), 'accuracy': acc,
        'y_true': np.asarray(trues), 'y_pred': np.asarray(preds),
    }


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix (Count)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='RdYlGn',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], vmin=0, vmax=100)
    axes[1].set_title('Confusion Matrix (Normalized %)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'confusion matrix saved: {save_path}')


class BaselineEncoder(nn.Module):
   
    def __init__(self, device_dims, d_model=D_MODEL, num_classes=16, dropout=DROPOUT,
                 pool='mean'):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.pool = pool
        feat_dim = int(sum(device_dims))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        feat = x.mean(dim=1) if self.pool != 'last' else x[:, -1]
        return self.classifier(self.dropout(feat))


class DynamicPriorGATEncoder(nn.Module):
    
    def __init__(self, device_dims, d_model=D_MODEL, num_classes=16,
                 gat_heads=GAT_HEADS, dropout=DROPOUT):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.device_projectors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, d_model), nn.LayerNorm(d_model), nn.GELU())
            for dim in device_dims
        ])
        prior_weight = torch.full((self.num_devices, self.num_devices), 0.1)
        for i, j in [(2, 3), (3, 2), (3, 4), (4, 3), (4, 5), (5, 4), (6, 3), (3, 6), (7, 3), (3, 7)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 1.0
        for i, j in [(0, 1), (1, 0), (0, 3), (3, 0), (1, 3), (3, 1)]:
            if i < self.num_devices and j < self.num_devices:
                prior_weight[i, j] = 0.2
        self.register_buffer('prior_weight', prior_weight)
        self.dynamic_prior_gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model), nn.GELU(), nn.Linear(d_model, 1), nn.Sigmoid())
        self.gat1 = GATv2Conv(d_model, d_model, heads=gat_heads, concat=False,
                              dropout=dropout, add_self_loops=False, edge_dim=1)
        self.n1 = nn.LayerNorm(d_model)
        out_dim = d_model * self.num_devices
        self.attn = nn.Linear(out_dim, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(out_dim * 2),
            nn.Linear(out_dim * 2, num_classes))

    def compute_dynamic_edges(self, pooled):
        B, N, D = pooled.shape
        device = pooled.device
        ei = torch.tensor(
            [[i, j] for i in range(N) for j in range(N) if i != j],
            dtype=torch.long, device=device).t().contiguous()
        normed = F.normalize(pooled, p=2, dim=-1)
        sim = torch.matmul(normed, normed.transpose(1, 2))
        dyn = torch.stack([sim[:, i, j] for i, j in zip(ei[0], ei[1])], dim=-1)
        fi, fj = pooled[:, ei[0]], pooled[:, ei[1]]
        alpha = self.dynamic_prior_gate(torch.cat([fi, fj], -1)).squeeze(-1)
        prior = self.prior_weight[ei[0], ei[1]].view(1, -1)
        agree = ((fi * fj).sum(-1) / (fi.norm(dim=-1) * fj.norm(dim=-1) + 1e-6)).clamp(0, 1)
        ew = ((alpha * prior + (1 - alpha) * dyn) * (0.5 + 0.5 * agree)).reshape(-1, 1)
        off = torch.arange(B, device=device) * N
        eib = (ei.unsqueeze(1) + off.view(1, -1, 1)).reshape(2, -1)
        return eib, ew

    def forward(self, x):
        B, T, _ = x.shape
        device_feats, split_idx = [], 0
        for proj, dim in zip(self.device_projectors, self.device_dims):
            device_feats.append(proj(x[:, :, split_idx:split_idx + dim]))
            split_idx += dim
        device_feats = torch.stack(device_feats, dim=2)
        pooled = device_feats.mean(1)
        edge_index, edge_weights = self.compute_dynamic_edges(pooled)
        h = pooled.reshape(-1, self.d_model)
        h = self.n1(h + self.gat1(h, edge_index, edge_attr=edge_weights))
        out_seq = device_feats + h.reshape(B, 1, self.num_devices, self.d_model)
        fused = out_seq.reshape(B, T, -1)
        w = torch.softmax(self.attn(fused).squeeze(-1), dim=1)
        feat = torch.cat([(fused * w.unsqueeze(-1)).sum(1), fused.mean(1)], dim=-1)
        return self.classifier(feat)


class SimpleMambaEncoder(nn.Module):
    
    def __init__(self, device_dims, d_model=D_MODEL, d_hidden=D_HIDDEN,
                 num_classes=16, dropout=DROPOUT):
        super().__init__()
        self.num_devices = len(device_dims)
        self.device_dims = device_dims
        self.d_model = d_model
        self.device_projectors = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, d_model), nn.LayerNorm(d_model), nn.GELU())
            for dim in device_dims
        ])
        self.dropout = nn.Dropout(dropout)
        mamba_dim = d_model * self.num_devices
        self.mamba = SimpleMamba(mamba_dim, d_hidden, kernel=3)
        self.attn = nn.Linear(mamba_dim, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(mamba_dim * 2),
            nn.Linear(mamba_dim * 2, num_classes))

    def forward(self, x):
        B, T, _ = x.shape
        device_feats, split_idx = [], 0
        for proj, dim in zip(self.device_projectors, self.device_dims):
            device_feats.append(proj(x[:, :, split_idx:split_idx + dim]))
            split_idx += dim
        fused = torch.cat(device_feats, dim=-1)
        fused = self.dropout(fused)
        fused = self.mamba(fused)
        w = torch.softmax(self.attn(fused).squeeze(-1), dim=1)
        feat = torch.cat([(fused * w.unsqueeze(-1)).sum(1), fused.mean(1)], dim=-1)
        return self.classifier(feat)


def build_ablation_model(cfg, device_dims, n_cls):
    if cfg['use_gat']:
        return DynamicPriorGATEncoder(
            device_dims=device_dims, d_model=D_MODEL, num_classes=n_cls,
            gat_heads=GAT_HEADS, dropout=DROPOUT
        )
    if cfg['use_mamba']:
        return SimpleMambaEncoder(
            device_dims=device_dims, d_model=D_MODEL, d_hidden=D_HIDDEN,
            num_classes=n_cls, dropout=DROPOUT
        )
    return BaselineEncoder(
        device_dims=device_dims, d_model=D_MODEL, num_classes=n_cls,
        dropout=DROPOUT, pool=cfg.get('pool', 'mean'),
    )


def train_one_epoch(model, loader, optimizer, device, n_cls, class_weight,
                    ema=None, use_kd=False, lam_kd=0.0, lam_f1=LAM_F1):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    pbar = tqdm(loader, desc='Train', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        t_logits = None
        if use_kd and ema is not None:
            with torch.no_grad():
                bak_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
                ema.copy_to(model)
                model.eval()
                t_logits = model(x)
                model.train()
                model.load_state_dict(bak_state, strict=True)
        s_logits = model(x)
        hard_loss = focal_loss(s_logits, y, weight=class_weight) + lam_f1 * soft_f1_loss(s_logits, y, n_cls)
        if use_kd and t_logits is not None and lam_kd > 0:
            loss = (1 - lam_kd) * hard_loss + lam_kd * kd_loss(s_logits, t_logits)
        else:
            loss = hard_loss
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if ema is not None:
            ema.update(model)
        bs = x.size(0)
        total_loss += loss.item() * bs
        all_preds.extend(s_logits.argmax(1).detach().cpu().numpy())
        all_labels.extend(y.cpu().numpy())
        pbar.set_postfix(loss=float(loss.item()))
    n = max(len(loader.dataset), 1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / n, f1


def run_single_ablation(cfg, train_ds, val_loader, test_loader, label_encoder,
                        class_weight, class_prior, train_loader):
    n_cls = len(label_encoder.classes_)
    model = build_ablation_model(cfg, train_ds.device_dims, n_cls).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*70}")
    print(f"[{cfg['name']}] GAT={cfg['use_gat']}  Mamba={cfg['use_mamba']}  "
          f"自蒸馏={cfg['use_mean_teacher']}  pool={cfg.get('pool')}  "
          f"参数量={n_params/1e6:.3f}M")
    print('='*70)

    ema = EMA(model, decay=EMA_DECAY) if cfg['use_mean_teacher'] else None
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2)

    ckpt_path = os.path.join(_ROOT, cfg['ckpt'])
    result_path = os.path.join(_ROOT, cfg['result'])
    cm_path = os.path.join(_ROOT, cfg['cm'])
    best_f1, best_epoch, bad = -1.0, 0, 0

    for epoch in range(1, EPOCHS + 1):
        lam_kd = consistency_rampup(epoch) if cfg['use_mean_teacher'] else 0.0
        print(f"\n--- Epoch {epoch}/{EPOCHS}  lam_kd={lam_kd:.3f} ---")
        tr_loss, tr_f1 = train_one_epoch(
            model, train_loader, optimizer, DEVICE, n_cls, class_weight,
            ema=ema, use_kd=cfg['use_mean_teacher'], lam_kd=lam_kd
        )
        if ema is not None:
            bak = {k: v.detach().clone() for k, v in model.state_dict().items()}
            ema.copy_to(model)
            val = evaluate(model, val_loader, DEVICE, label_encoder, desc='Valid')
            model.load_state_dict(bak)
            save_state = {k: v.cpu().clone() for k, v in ema.shadow.items()}
        else:
            val = evaluate(model, val_loader, DEVICE, label_encoder, desc='Valid')
            save_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(f'Train Loss={tr_loss:.4f}  Train F1={tr_f1:.4f}  Val Macro-F1={val["macro_f1"]:.4f}')
        if val['macro_f1'] > best_f1 + 1e-4:
            best_f1, best_epoch, bad = val['macro_f1'], epoch, 0
            torch.save({
                'model_state': save_state,
                'epoch': best_epoch,
                'val_macro_f1': best_f1,
                'classes': list(label_encoder.classes_),
                'device_dims': list(train_ds.device_dims),
                'ablation': cfg['tag'],
                'use_gat': cfg['use_gat'],
                'use_mamba': cfg['use_mamba'],
                'use_mean_teacher': cfg['use_mean_teacher'],
                'pool': cfg.get('pool'),
                'd_model': D_MODEL, 'd_hidden': D_HIDDEN, 'gat_heads': GAT_HEADS,
                'dropout': DROPOUT, 'clf': CLF_TAG, 'lr': LR, 'batch_size': BATCH_SIZE,
                'scheduler': SCHEDULER_NAME,
                'class_prior': class_prior.cpu(),
            }, ckpt_path)
            print(f'  >> 最优模型已保存，Val Macro-F1={best_f1:.4f}')
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'早停触发（patience={PATIENCE}）')
                break
        scheduler.step(val['macro_f1'])
        print(f'  lr={optimizer.param_groups[0]["lr"]:.2e}')

    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'], strict=True)
    val_logits, val_y = collect_logits(model, val_loader, DEVICE)
    adj_tau, _ = tune_logit_adjustment(val_logits, val_y, class_prior, list(range(n_cls)))
    ckpt['adj_tau'] = adj_tau
    torch.save(ckpt, ckpt_path)

    test = evaluate(model, test_loader, DEVICE, label_encoder, desc='Test',
                    class_prior=class_prior, adj_tau=adj_tau)
    val = evaluate(model, val_loader, DEVICE, label_encoder, desc='Valid',
                   class_prior=class_prior, adj_tau=adj_tau)

    with open(result_path, 'w', encoding='utf-8') as f:
        f.write(f"消融组: {cfg['name']}\n")
        f.write(f"GAT: {cfg['use_gat']}\n")
        f.write(f"Mamba: {cfg['use_mamba']}\n")
        f.write(f"Mean-Teacher自蒸馏: {cfg['use_mean_teacher']}\n")
        f.write(f"pool: {cfg.get('pool')}\n")
        f.write(f"参数量(M): {n_params/1e6:.6f}\n")
        f.write(f"最优轮次: {best_epoch}\n")
        f.write(f"adj_tau: {adj_tau:.6f}\n")
        f.write(f"Val Macro-F1: {val['macro_f1']:.6f}\n")
        f.write(f"Test Macro-F1: {test['macro_f1']:.6f}\n")
        f.write(f"Test Accuracy: {test['accuracy']:.6f}\n\n")
        f.write('===== TEST 分类报告 =====\n')
        f.write(test['report'] + '\n')

    plot_confusion_matrix(test['y_true'], test['y_pred'],
                          list(label_encoder.classes_), cm_path)
    print(f"测试完成 | Test Macro-F1={test['macro_f1']:.4f} | Test Acc={test['accuracy']:.4f} | adj_tau={adj_tau:.3f}")
    print(f'结果已保存: {result_path}')
    return dict(
        name=cfg['name'],
        use_gat=cfg['use_gat'],
        use_mamba=cfg['use_mamba'],
        use_mean_teacher=cfg['use_mean_teacher'],
        params_M=n_params / 1e6,
        best_epoch=best_epoch,
        val_f1=float(val['macro_f1']),
        test_f1=float(test['macro_f1']),
        test_acc=float(test['accuracy'])
    )


def load_saved_ablation(cfg):
    result_path = os.path.join(_ROOT, cfg['result'])
    kv = {}
    with open(result_path, encoding='utf-8') as f:
        for line in f:
            if ':' in line:
                k, v = line.split(':', 1)
                kv[k.strip()] = v.strip()
    print(f"[{cfg['name']}] 复用已有结果: {result_path}")
    use_gat = kv.get('GAT', str(cfg['use_gat']))
    use_mamba = kv.get('Mamba', str(cfg['use_mamba']))
    use_mt = kv.get('Mean-Teacher自蒸馏', str(cfg['use_mean_teacher']))
    return dict(
        name=cfg['name'],
        use_gat=use_gat.lower() == 'true' if isinstance(use_gat, str) else bool(use_gat),
        use_mamba=use_mamba.lower() == 'true' if isinstance(use_mamba, str) else bool(use_mamba),
        use_mean_teacher=use_mt.lower() == 'true' if isinstance(use_mt, str) else bool(use_mt),
        params_M=float(kv['参数量(M)']),
        best_epoch=int(kv['最优轮次']),
        val_f1=float(kv['Val Macro-F1']),
        test_f1=float(kv['Test Macro-F1']),
        test_acc=float(kv['Test Accuracy']),
    )


def main():
    print('=' * 70)
    print('三组消融实验 | 超参与 UltraLite 主实验严格对齐 | 流程对齐目录2')
    print(f'设备: {DEVICE} | 数据: {DATA_CSV}')
    print(f'窗口={WINDOW_SIZE} 步长={TRAIN_STEP}/{EVAL_STEP} 批次={BATCH_SIZE} '
          f'学习率={LR} 早停={PATIENCE} dropout={DROPOUT}')
    print('=' * 70)

    df = pd.read_csv(DATA_CSV, low_memory=False)
    label_encoder = LabelEncoder()
    df['label_encoded'] = label_encoder.fit_transform(df['final_type'])
    n_cls = len(label_encoder.classes_)
    normal_idx = (int(np.where(label_encoder.classes_ == 'Normal')[0][0])
                  if 'Normal' in set(label_encoder.classes_) else None)
    print(f'\n类别数: {n_cls}')
    print(f'类别列表: {list(label_encoder.classes_)}')

    df_train = df[df.split == 'train'].reset_index(drop=True)
    df_val = df[df.split == 'val'].reset_index(drop=True)
    df_test = df[df.split == 'test'].reset_index(drop=True)

    train_ds = EdgeIIoTFusionDataset(
        df_train, window_size=WINDOW_SIZE, step=TRAIN_STEP, fit_scaler=True,
        max_windows=MAX_TRAIN, min_purity=MIN_PURITY, normal_idx=normal_idx)
    val_ds = EdgeIIoTFusionDataset(
        df_val, window_size=WINDOW_SIZE, step=EVAL_STEP, scaler=train_ds.scaler,
        max_windows=MAX_EVAL, min_purity=MIN_PURITY, normal_idx=normal_idx)
    test_ds = EdgeIIoTFusionDataset(
        df_test, window_size=WINDOW_SIZE, step=EVAL_STEP, scaler=train_ds.scaler,
        max_windows=MAX_EVAL, min_purity=MIN_PURITY, normal_idx=normal_idx)

    counts = np.maximum(np.bincount(train_ds.labels, minlength=n_cls).astype(np.float64), 1.0)
    sw = 1.0 / np.power(counts[train_ds.labels], 0.35)
    sw = sw / sw.mean()
    sampler = torch.utils.data.WeightedRandomSampler(torch.DoubleTensor(sw), len(train_ds), True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    cw = np.clip(np.sqrt(counts.sum() / (n_cls * counts)), 0.5, 2.0)
    class_weight = torch.tensor(cw, dtype=torch.float32, device=DEVICE)
    class_prior = torch.tensor(counts / counts.sum(), dtype=torch.float32)

    results = []
    for cfg in ABLATION_CONFIGS:
        result_path = os.path.join(_ROOT, cfg['result'])
        reuse = (RETRAIN_TAGS is not None and cfg['tag'] not in RETRAIN_TAGS
                 and os.path.isfile(result_path))
        if reuse:
            results.append(load_saved_ablation(cfg))
            continue
        results.append(run_single_ablation(
            cfg, train_ds, val_loader, test_loader, label_encoder,
            class_weight, class_prior, train_loader
        ))

    print('\n' + '=' * 70)
    print('消融实验汇总')
    print('=' * 70)
    header = (f"{'组别':<22} {'GAT':<6} {'Mamba':<6} {'自蒸馏':<8} {'参数量(M)':>10} "
              f"{'最优轮':>6} {'Val-F1':>8} {'Test-F1':>8} {'Test-Acc':>8}")
    print(header)
    print('-' * 90)
    with open(SUMMARY_TXT, 'w', encoding='utf-8') as f:
        f.write('消融实验汇总（与UltraLite主实验超参对齐）\n')
        f.write('A0=纯基线 | A1=先验动态门控GAT | A2=SimpleMamba+自蒸馏\n')
        f.write(f'window={WINDOW_SIZE} train_step={TRAIN_STEP} eval_step={EVAL_STEP} ')
        f.write(f'batch={BATCH_SIZE} lr={LR} patience={PATIENCE} dropout={DROPOUT}\n\n')
        f.write(header + '\n')
        f.write('-' * 90 + '\n')
        for r in results:
            line = (f"{r['name']:<22} {str(r['use_gat']):<6} {str(r['use_mamba']):<6} "
                    f"{str(r['use_mean_teacher']):<8} "
                    f"{r['params_M']:>10.3f} {r['best_epoch']:>6d} {r['val_f1']:>8.4f} "
                    f"{r['test_f1']:>8.4f} {r['test_acc']:>8.4f}")
            print(line)
            f.write(line + '\n')
    print(f'\n汇总结果已保存: {SUMMARY_TXT}')
    print('所有模型权重、分类报告、混淆矩阵均已保存到当前目录')


main()


## 跨协议融合方法对比（4 种：Ours + 3 替代）

同一 UltraLite 骨干，只替换 **共识构造 + 融合强度**：

| 方法 | 说明 |
|------|------|
| **Ours** | 活性加权共识 + 可学习软融合 γ |
| Uniform | 均匀共识 + 可学习 γ（去掉活性加权） |
| Low-Fixed-γ | 活性加权 + **固定过低** γ=0.05（缺失时无法像 Ours 增大 γ） |
| Private-only | γ≡0，不做跨协议融合 |

压测：去掉最弱 0–4 协议 → `photo/fusion_drop_protocol.png`（x=丢弃数 0–4）

出图：`photo/plot_fusion_drop_aprf.py`（与 ph 布局/字体一致）。重训：`FORCE_FUSION_TRAIN=1`


In [ ]:
# ===== 跨协议融合对比图：Dropped protocols 0–4 =====
import os, runpy
from pathlib import Path
from IPython.display import Image, display

_ROOT = Path(r'E:/apt/YES/1')
if not (_ROOT / 'photo' / 'plot_fusion_drop_aprf.py').is_file():
    _ROOT = Path('.').resolve()
os.chdir(_ROOT)
os.environ.setdefault('FORCE_FUSION_TRAIN', '0')
os.environ.setdefault('FORCE_FUSION_TRAIN_OURS', '0')
os.environ.setdefault('FORCE_FUSION_ALT', '0')
if os.environ.get('FORCE_FUSION_TRAIN') == '1' or os.environ.get('FORCE_FUSION_ALT') == '1':
    runpy.run_path(str(_ROOT / '_run_fusion_method_compare.py'), run_name='__main__')
runpy.run_path(str(_ROOT / 'photo' / 'plot_fusion_drop_aprf.py'), run_name='__main__')
p = _ROOT / 'photo' / 'fusion_drop_protocol.png'
if p.is_file():
    display(Image(filename=str(p)))


## 跨设备融合方法对比（4 种）

数据：`E:\apt\YES\2\iot_fusion_weather_all.csv`（7 路设备融合）

| 方法 | 说明 |
|------|------|
| **Ours** | 活性加权共识 + 可学习软融合 γ |
| Uniform | 均匀共识 + 可学习 γ（无活性加权） |
| Low-Fixed-γ | 活性加权 + **固定过低** γ=0.05（缺失时无法像 Ours 增大 γ） |
| Private-only | γ≡0（无跨设备融合） |

压测：去掉最弱 0–4 设备 → `photo/fusion_drop_device.png`（x=丢弃数 0–4）

出图：`photo/plot_fusion_drop_aprf.py`（与 ph 布局/字体一致）。重训：`FORCE_FUSION_DEVICE_TRAIN=1`

In [ ]:
# ===== 跨设备融合对比图：Dropped devices 0–4 =====
import os, runpy
from pathlib import Path
from IPython.display import Image, display

_ROOT = Path(r'E:/apt/YES/1')
if not (_ROOT / 'photo' / 'plot_fusion_drop_aprf.py').is_file():
    _ROOT = Path('.').resolve()
os.chdir(_ROOT)
os.environ.setdefault('FORCE_FUSION_DEVICE_TRAIN', '0')  # 1=重训全部
os.environ.setdefault('FORCE_FUSION_DEVICE_OURS', '0')    # 1=仅重训 Ours
os.environ.setdefault('FORCE_FUSION_DEVICE_ALT', '0')
if os.environ.get('FORCE_FUSION_DEVICE_TRAIN') == '1' or os.environ.get('FORCE_FUSION_DEVICE_OURS') == '1':
    runpy.run_path(str(_ROOT / '_run_fusion_method_compare_device.py'), run_name='__main__')
runpy.run_path(str(_ROOT / 'photo' / 'plot_fusion_drop_aprf.py'), run_name='__main__')
p = _ROOT / 'photo' / 'fusion_drop_device.png'
if p.is_file():
    display(Image(filename=str(p)))

## 自蒸馏超参敏感性分析（Mean-Teacher）

在 **UltraLite** 骨干上，对自蒸馏关键超参做 **单因素（OAT）** 扫描（其余固定为默认值）。数据改为 **YES/2 `iot_fusion_weather_all.csv`**（7 设备）：

| 超参 | 默认 | 扫描范围 | 含义 |
|------|------|----------|------|
| `lam_kd` | 0.65 | 0.35 / 0.50 / 0.65 / 0.80 | 软标签最大权重（硬损=1−λ；） |
| `ema_decay` | 0.999 | 0.990 / 0.995 / 0.999 / 0.9995 | EMA 教师平滑系数 |
| `ramp_epochs` | 10 | 0 / 5 / 10 / 20 | consistency 斜坡长度 |
| `kd_temp` | 2.0 | 1.0 / 2.0 / 4.0 | KL 温度 \(T\) |

- 对齐 YES/2 UltraLite：`window=128`，`train_step=1`，`eval_step=2`，`batch=64`，`lr=1e-3`，`EPOCHS=50`，`PATIENCE=5`，时间块分层划分
- 训练配方：Gaussian ramp-up、focal+soft-F1、EMA 教师、默认点只训一次后复用到各轴
- **不覆盖** `best_gat_mamba_ultralite.pth`
- 输出：`selfkd_sensitivity_{lam,ema,ramp,temp}.{csv,txt}`、`photo/selfkd_sensitivity_*.{png,pdf}`、汇总 `selfkd_sensitivity_summary.txt`
- 支持断点续跑（`selfkd_sensitivity_cache.csv`）；实现见 `_run_selfkd_sensitivity.py`


In [ ]:
# ===== Mean-Teacher 自蒸馏超参敏感性分析（可单独运行）=====
# 数据: YES/2 iot_fusion_weather_all.csv；配方对齐该目录 UltraLite
# 实现: _run_selfkd_sensitivity.py（不覆盖 best_gat_mamba_ultralite.pth）
from pathlib import Path
_p = Path('_run_selfkd_sensitivity.py')
if not _p.is_file():
    _p = Path('/root/1/YES/1/_run_selfkd_sensitivity.py')
assert _p.is_file(), _p
exec(compile(_p.read_text(encoding='utf-8'), str(_p), 'exec'))
